## 1. Models

## Models Used for Testing

To evaluate the performance of different Large Language Models (LLMs), we selected 15 open-source models from :contentReference[oaicite:0]{index=0} and divided them into five categories based on parameter size. Each category contains three models, allowing us to compare how model size affects performance in the calorie estimation task.

### Ultra-Light Models (< 1B parameters)

These models are very small and lightweight, making them fast and suitable for limited hardware environments.

- "Qwen/Qwen2.5-0.5B-Instruct" – A 0.5B parameter instruction-tuned model.
- "HuggingFaceTB/SmolLM2-360M-Instruct" – A compact instruction model designed for efficient local inference.
- "EleutherAI/pythia-410m" – A small research-focused language model developed by EleutherAI.

### Small Models (1B – 3B parameters)

These models provide improved reasoning ability while maintaining relatively low computational cost.

- "TinyLlama/TinyLlama-1.1B-Chat-v1.0" – A small chat-oriented model derived from LLaMA-style architectures.
- "stabilityai/stablelm-2-zephyr-1_6b" – An instruction-tuned conversational model optimized for efficiency.
- "microsoft/phi-2" – A compact reasoning model developed by Microsoft.

### Medium Models (3B – 5B parameters)

These models balance performance and resource consumption.

- "Qwen/Qwen2.5-3B-Instruct" – A mid-sized instruction-following model.
- "microsoft/Phi-3-mini-4k-instruct" – A lightweight yet powerful model optimized for instruction tasks.
- "stabilityai/stablelm-zephyr-3b" – A conversational model with improved reasoning compared to smaller models.

### Large Models (5B – 7B parameters)

These models provide stronger reasoning capability but require more computational resources.

- "HuggingFaceH4/zephyr-7b-beta" – A high-quality instruction-tuned dialogue model.
- "NousResearch/Nous-Hermes-2-Mistral-7B-DPO" – A Mistral-based model optimized for instruction following.
- "tiiuae/falcon-7b-instruct" – A widely used open-source instruction model.

### Extra-Large Models (7B – 10B+ parameters)

These models have larger parameter sizes and are expected to provide stronger reasoning performance.

- "01-ai/Yi-1.5-9B-Chat" – A multilingual chat model with strong reasoning ability.
- "upstage/SOLAR-10.7B-Instruct-v1.0" – A high-performance instruction model designed for advanced reasoning.
- "lmsys/vicuna-7b-v1.5" – A conversational model derived from LLaMA-based training.

These models were selected to represent a wide spectrum of parameter sizes, allowing performance comparison in the calorie estimation task.

### 2. Test the model

We use prompt "How many calories does a average 35-year-old woman who weighs 70kg and is 180 cm tall need per day?" for every model and see the results with time spent for each model.

In [3]:

import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
import time

In [ ]:




models_to_test = [
    # Ultra-Light (< 1B parameters)
    "Qwen/Qwen2.5-0.5B-Instruct",
    "HuggingFaceTB/SmolLM2-360M-Instruct",
    "EleutherAI/pythia-410m",

    # Small (1B - 3B parameters)
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "stabilityai/stablelm-2-zephyr-1_6b",
    "microsoft/phi-2",

    # Medium (3B - 5B parameters)
    "Qwen/Qwen2.5-3B-Instruct",
    "microsoft/Phi-3-mini-4k-instruct",
    "stabilityai/stablelm-zephyr-3b",

    # Large (5B - 7B parameters)
    "HuggingFaceH4/zephyr-7b-beta",
    "NousResearch/Nous-Hermes-2-Mistral-7B-DPO",
    "tiiuae/falcon-7b-instruct",

    # Extra-Large (7B - 10B+ parameters)
    "01-ai/Yi-1.5-9B-Chat",
    "upstage/SOLAR-10.7B-Instruct-v1.0",
    "lmsys/vicuna-7b-v1.5"
]

test_prompts = ["How many calories does a average 35-year-old woman who weighs 70kg and is 180 cm tall need per day?"]

results = {}

for model_id in models_to_test:
    print(f"\n--- Testing Model: {model_id} ---")
    try:
       
       


        try:
            
            tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
            model = AutoModelForCausalLM.from_pretrained(
                model_id, 
                device_map="auto", 
                dtype=torch.float16,
                trust_remote_code=False
            )
        except Exception as e:
            print(f"    [Native load failed, trying remote code...] {e}")
           
            tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
            model = AutoModelForCausalLM.from_pretrained(
                model_id, 
                device_map="auto", 
                dtype=torch.float16,
                trust_remote_code=True
            )

            tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, use_fast=False)

        # model = AutoModelForCausalLM.from_pretrained(
        #     model_id, 
        #     device_map="auto", 
        #     dtype=torch.float16,
        #     trust_remote_code=True
        # )
        
        results[model_id] = []
        
        
        for prompt in test_prompts:
            formatted_prompt = f"Question: {prompt}\nAnswer:"
            inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
            
            start_time = time.time()
            
            outputs = model.generate(**inputs, max_new_tokens=300) 
            end_time = time.time()
            
            
            response = tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            results[model_id].append({
                "prompt": prompt,
                "response": response,
                "time_taken": round(end_time - start_time, 2)
               
            })
            print(f"{response}\nTime: {round(end_time - start_time, 2)} \n")
            
    except Exception as e:
        print(f"Failed to load or run {model_id}: {e}")
        
    finally:

        if 'model' in locals():
            del model
        if 'tokenizer' in locals():
            del tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


--- Testing Model: Qwen/Qwen2.5-0.5B-Instruct ---


Some parameters are on the meta device because they were offloaded to the cpu.


Question: How many calories does a average 35-year-old woman who weighs 70kg and is 180 cm tall need per day?
Answer: The average person needs about 2400-3000 calories per day. To calculate this, we use the formula: number of calories = weight (kg) + height (meters) x 100 + age (years) x 0.69. In this case, that would be: 70 kg + 180 m x 0.69 = 2400 - 3000 = 600 calories.
Therefore, the answer is 600.
Time: 16.22 


--- Testing Model: HuggingFaceTB/SmolLM2-360M-Instruct ---
Question: How many calories does a average 35-year-old woman who weighs 70kg and is 180 cm tall need per day?
Answer:
A 35-year-old woman weighs 70 kg and is 180 cm tall.
The weight in kilograms is 70 kg.
The height in centimeters is 180 cm.
The weight in kilograms is 70 kg.
The height in centimeters is 180 cm.
The weight in calories is 70 kg * 1000 = 70000 calories.
The height in calories is 180 cm * 1000 = 180000 calories.
The total calories needed per day is 70000 + 180000 = 250000 calories.
The average 35-year-o

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Question: How many calories does a average 35-year-old woman who weighs 70kg and is 180 cm tall need per day?
Answer: The answer is:

The average weight of a woman is 70kg.
The average height of a woman is 180cm.
The average height of a man is 180cm.

The average weight of a man is 70kg.
The average height of a man is 180cm.

The average weight of a woman is 70kg.
The average height of a woman is 180cm.

The average weight of a man is 70kg.
The average height of a man is 180cm.

The average weight of a woman is 70kg.
The average height of a woman is 180cm.

The average weight of a man is 70kg.
The average height of a man is 180cm.

The average weight of a woman is 70kg.
The average height of a woman is 180cm.

The average weight of a man is 70kg.
The average height of a man is 180cm.

The average weight of a woman is 70kg.
The average height of a woman is 180cm.

The average weight of a man is 70kg.
The average height of a man is 180cm.

The average weight of a woman is 70kg.
The avera

...

Some parameters are on the meta device because they were offloaded to the disk and cpu.


Question: How many calories does a average 35-year-old woman who weighs 70kg and is 180 cm tall need per day?
Answer: To calculate the daily caloric needs of an individual, we can use the Harris-Benedict equation.
For a woman, the equation is:
Basal Metabolic Rate (BMR) = 655 + (9.6 * weight in kg) + (1.8 * height in cm) - (4.7 * age in years)
A 35-year-old woman with a weight of 70 kg and a height of 180 cm would have a BMR of:
BMR = 655 + (9.6 * 70) + (1.8 * 180) - (4.7 * 35)
BMR = 655 + 672 + 324 - 159
BMR = 1302
To find the total daily caloric needs, we need to multiply the BMR by an activity factor.
For a sedentary lifestyle (little or no exercise), the activity factor is 1.2.
So, the total daily caloric needs would be:
Total Calories = BMR * Activity Factor
Total Calories = 1302 * 1.2
Total Calories = 1562.4
Rounding to the nearest whole number, a 35-year-old woman with the given information
Time: 3142.94 


--- Testing Model: lmsys/vicuna-7b-v1.5 ---


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Question: How many calories does a average 35-year-old woman who weighs 70kg and is 180 cm tall need per day?
Answer: To determine how many calories a 35-year-old woman who weighs 70kg and is 180cm tall needs per day, we need to consider her age, weight, height, and activity level.

According to the USDA's MyPlate guidelines, a woman in this age group who weighs 70kg and is 180cm tall needs around 2000-2500 calories per day to maintain her weight. However, this can vary depending on her activity level and other factors.

If the woman is sedentary (little to no exercise), she may need closer to 2000 calories per day. If she is moderately active, she may need around 2200-2500 calories per day.

It's important to note that everyone's calorie needs are different, and it's best to consult with a healthcare professional or a registered dietitian to determine the appropriate calorie intake for your individual needs.
Time: 520.88 



In [6]:
import json

with open('model_matrix_results.json', 'w') as f:
    json.dump(results, f, indent=4)



Results successfully saved to model_matrix_results.json!


## 3. Prompt Engineering

In [29]:
final_result = {}


for model_id in models_to_test:
    final_result[model_id] = results[model_id][0]["response"].split("Answer:")[1].replace("\n", " ").strip()

    print(model_id ,":\n","Result:",final_result[model_id],f"\nTime: {results[model_id][0]['time_taken']} seconds\n---------------------------------------------------------------------------------")




Qwen/Qwen2.5-0.5B-Instruct :
 Result: The average person needs about 2400-3000 calories per day. To calculate this, we use the formula: number of calories = weight (kg) + height (meters) x 100 + age (years) x 0.69. In this case, that would be: 70 kg + 180 m x 0.69 = 2400 - 3000 = 600 calories. Therefore, the answer is 600. 
Time: 16.22 seconds
---------------------------------------------------------------------------------
HuggingFaceTB/SmolLM2-360M-Instruct :
 Result: A 35-year-old woman weighs 70 kg and is 180 cm tall. The weight in kilograms is 70 kg. The height in centimeters is 180 cm. The weight in kilograms is 70 kg. The height in centimeters is 180 cm. The weight in calories is 70 kg * 1000 = 70000 calories. The height in calories is 180 cm * 1000 = 180000 calories. The total calories needed per day is 70000 + 180000 = 250000 calories. The average 35-year-old woman needs 250000 calories per day. #### 250000 The answer is: 250000 
Time: 13.38 seconds
---------------------------

Since we can see that some models spend too much time, we want to minimize the inference time while still maintaining good results. Therefore, we selected two models, `Qwen/Qwen2.5-0.5B-Instruct` and `TinyLlama/TinyLlama-1.1B-Chat-v1.0`, for prompt engineering.

In [7]:
def zero_shot(prompt):
    return f"""
{prompt}
Respond in this exact format:
BMR: X kcal
TDEE (sedentary): X kcal
TDEE (moderately active): X kcal
TDEE (very active): X kcal

Output:
"""

def one_shot(prompt):
    return f"""

You are a calorie estimation assistant. Use the Mifflin-St Jeor equation.
Always return all four activity levels in the exact format below.

Example:
Input: 25-year-old man, 80 kg, 175 cm
Output:
  BMR: 1,877 kcal
  TDEE (sedentary): 2,252 kcal
  TDEE (moderately active): 2,598 kcal
  TDEE (very active): 2,944 kcal

Now answer:
{prompt}
Output:

"""

def few_shot(prompt):
    return f"""
You estimate daily calorie needs using the Mifflin-St Jeor equation.
Always return results in the structured format shown below.

Example 1:
Input: 25-year-old man, 80 kg, 175 cm
Output:
  BMR: 1,877 kcal
  TDEE (sedentary): 2,252 kcal
  TDEE (moderately active): 2,598 kcal
  TDEE (very active): 2,944 kcal

Example 2:
Input: 45-year-old woman, 60 kg, 160 cm
Output:
  BMR: 1,304 kcal
  TDEE (sedentary): 1,565 kcal
  TDEE (moderately active): 1,805 kcal
  TDEE (very active): 2,046 kcal

Example 3:
Input: 30-year-old man, 90 kg, 185 cm
Output:
  BMR: 2,040 kcal
  TDEE (sedentary): 2,448 kcal
  TDEE (moderately active): 2,824 kcal
  TDEE (very active): 3,200 kcal

Now answer:
{prompt}
Output:

"""

def many_shot(prompt):
    return f"""
You estimate daily calorie needs using the Mifflin-St Jeor equation.
Always return results in the structured format shown below.

Example 1:
Input: 25-year-old man, 80 kg, 175 cm
Output:
  BMR: 1,877 kcal
  TDEE (sedentary): 2,252 kcal
  TDEE (moderately active): 2,598 kcal
  TDEE (very active): 2,944 kcal

Example 2:
Input: 45-year-old woman, 60 kg, 160 cm
Output:
  BMR: 1,304 kcal
  TDEE (sedentary): 1,565 kcal
  TDEE (moderately active): 1,805 kcal
  TDEE (very active): 2,046 kcal

Example 3:
Input: 30-year-old man, 90 kg, 185 cm
Output:
  BMR: 2,040 kcal
  TDEE (sedentary): 2,448 kcal
  TDEE (moderately active): 2,824 kcal
  TDEE (very active): 3,200 kcal

Example 4:
Input: 40-year-old woman, 70 kg, 170 cm
Output:
  BMR: 1,542 kcal
  TDEE (sedentary): 1,851 kcal
  TDEE (moderately active): 2,134 kcal
  TDEE (very active): 2,418 kcal

Example 5:
Input: 55-year-old man, 85 kg, 175 cm
Output:
  BMR: 1,771 kcal
  TDEE (sedentary): 2,125 kcal
  TDEE (moderately active): 2,452 kcal
  TDEE (very active): 2,779 kcal

Now answer:
{prompt}
Output:
"""


In [8]:
models_to_test = [

    "Qwen/Qwen2.5-0.5B-Instruct",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "stabilityai/stablelm-2-zephyr-1_6b",
 
]
shots = ["zero shot", "one shot", "few shot", "many shot"]

based_promp = "How many calories does a average 35-year-old woman who weighs 70kg and is 180 cm tall need per day?"
test_prompts = [zero_shot(based_promp), one_shot(based_promp), few_shot(based_promp), many_shot(based_promp)]

test_results = {}

for model_id in models_to_test:
    print(f"\n--- Testing Model: {model_id} ---")
    try:
       
    

      
            
        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            device_map="auto", 
            dtype=torch.float16,
            trust_remote_code=False
        )


   
        
        
        
        test_results[model_id] = []
        for shot, prompt in zip(shots, test_prompts):
            
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            
            start_time = time.time()
            
            outputs = model.generate(**inputs, max_new_tokens=300) 
            end_time = time.time()
            
            
            response = tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            test_results[model_id].append({
                "prompt": prompt,
                "response": response,
                "time_taken": round(end_time - start_time, 2)
               
            })
            print(shot, "\n")
            print(f"Response: {response}\nTime: {round(end_time - start_time, 2)} \n")
            print("---------------------------------------------------------------------------------------------------")
            
    except Exception as e:
        print(f"Failed to load or run {model_id}: {e}")
        
    finally:

        if 'model' in locals():
            del model
        if 'tokenizer' in locals():
            del tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


--- Testing Model: Qwen/Qwen2.5-0.5B-Instruct ---
zero shot 

Response: 
How many calories does a average 35-year-old woman who weighs 70kg and is 180 cm tall need per day?
Respond in this exact format:
BMR: X kcal
TDEE (sedentary): X kcal
TDEE (moderately active): X kcal
TDEE (very active): X kcal

Output:
X = 2400 kcal

Therefore, the answer is 2400 kcal. 

(BMR: 66 + (10 * height) - (10 * weight) = 148
TDEE (sedentary): 2400 kcal
TDEE (moderately active): 2400 kcal
TDEE (very active): 2400 kcal)

The problem asks for the total daily energy expenditure (TDEE) based on age, weight, and height. Since no other factors are mentioned, we assume these values to be typical for an adult woman.

For simplicity, let's calculate the TDEE assuming she is sedentary:

1. BMR: 66 + (10 * 170) = 2400 kcal
2. TDEE (sedentary): 2400 kcal

Therefore, the answer is 2400 kcal. 

(BMR: 66 + (10 * height) - (10 * weight) = 148
TDEE (sedentary): 2400 kcal
TDEE (moderately active): 2400 kcal
TDEE (very acti

Setting `pad_token_id` to `eos_token_id`:100257 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100257 for open-end generation.


zero shot 

Response: 
How many calories does a average 35-year-old woman who weighs 70kg and is 180 cm tall need per day?
Respond in this exact format:
BMR: X kcal
TDEE (sedentary): X kcal
TDEE (moderately active): X kcal
TDEE (very active): X kcal

Output:
BMR: 1360 kcal
TDEE (sedentary): 1800 kcal
TDEE (moderately active): 2160 kcal
TDEE (very active): 2520 kcal
Time: 1.84 

---------------------------------------------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:100257 for open-end generation.


one shot 

Response: 

You are a calorie estimation assistant. Use the Mifflin-St Jeor equation.
Always return all four activity levels in the exact format below.

Example:
Input: 25-year-old man, 80 kg, 175 cm
Output:
  BMR: 1,877 kcal
  TDEE (sedentary): 2,252 kcal
  TDEE (moderately active): 2,598 kcal
  TDEE (very active): 2,944 kcal

Now answer:
How many calories does a average 35-year-old woman who weighs 70kg and is 180 cm tall need per day?
Output:

BMR: 1,445 kcal
TDEE (sedentary): 1,820 kcal
TDEE (moderately active): 2,205 kcal
TDEE (very active): 2,580 kcal
Time: 1.77 

---------------------------------------------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:100257 for open-end generation.


few shot 

Response: 
You estimate daily calorie needs using the Mifflin-St Jeor equation.
Always return results in the structured format shown below.

Example 1:
Input: 25-year-old man, 80 kg, 175 cm
Output:
  BMR: 1,877 kcal
  TDEE (sedentary): 2,252 kcal
  TDEE (moderately active): 2,598 kcal
  TDEE (very active): 2,944 kcal

Example 2:
Input: 45-year-old woman, 60 kg, 160 cm
Output:
  BMR: 1,304 kcal
  TDEE (sedentary): 1,565 kcal
  TDEE (moderately active): 1,805 kcal
  TDEE (very active): 2,046 kcal

Example 3:
Input: 30-year-old man, 90 kg, 185 cm
Output:
  BMR: 2,040 kcal
  TDEE (sedentary): 2,448 kcal
  TDEE (moderately active): 2,824 kcal
  TDEE (very active): 3,200 kcal

Now answer:
How many calories does a average 35-year-old woman who weighs 70kg and is 180 cm tall need per day?
Output:

BMR: 1,990 kcal
TDEE (sedentary): 2,455 kcal
TDEE (moderately active): 2,900 kcal
TDEE (very active): 3,345 kcal
Time: 1.77 

--------------------------------------------------------------

In [ ]:
def calculate_bmr(weight_kg, height_cm, age_years, sex):
    """
    Calculates Basal Metabolic Rate (BMR) using the Mifflin-St Jeor Equation.
    """
    
    base_bmr = (10 * weight_kg) + (6.25 * height_cm) - (5 * age_years)
    
    if sex.lower() == 'male':
        return base_bmr + 5
    elif sex.lower() == 'female':
        return base_bmr - 161
    else:
        raise ValueError("Sex must be 'male' or 'female'")

def calculate_tdee(bmr, activity_level):
    """
    Calculates Total Daily Energy Expenditure (TDEE) based on activity level.
    """
    activity_factors = {
        "sedentary": 1.2,          
        "lightly active": 1.375,    
        "moderately active": 1.55, 
        "very active": 1.725,      
        "extra active": 1.9         
    }
    
    factor = activity_factors.get(activity_level.lower())
    if factor is None:
        raise ValueError("Invalid activity level provided.")
    
    return bmr * factor

print("BMR: ",calculate_bmr(70, 180, 35,"female"))
print("TDEE (sedentary): ",calculate_tdee(calculate_bmr(70, 180, 35,"female"), "sedentary"))
print("TDEE (moderately active): ",calculate_tdee(calculate_bmr(70, 180, 35,"female"), "moderately active"))
print("TDEE (very active): ",calculate_tdee(calculate_bmr(70, 180, 35,"female"), "very active"))

BMR:  1489.0
TDEE (sedentary):  1786.8
TDEE (moderately active):  2307.9500000000003
TDEE (very active):  2568.525


Based on the results, the one-shot and zero-shot answer from the `stabilityai/stablelm-2-zephyr-1_6` model yields the best result because it provides the output format that we want, spend least time, and the results are also the closest to the actual values.

## 4. Evaluation

## 1. Instruction Adherence (Formatting)



**Qwen 0.5B:**  
**Moderate.** It consistently provided the requested structure at the top of its response, but it completely failed to stop. It appended unprompted text, step-by-step breakdowns, and even generated Python scripts in the Few/Many-shot tests.

**TinyLlama 1.1B:**  
**Poor.** While Zero-shot was relatively contained (only adding a short explanation), introducing examples caused it to spiral into infinite loops, hallucinating new prompts and repeating the same answers endlessly.

**StableLM 2 Zephyr 1.6B:**  
**Excellent.** In Zero-shot, One-shot, and Few-shot, it returned the exact four lines requested and stopped immediately. No extra text, no explanations, and no code. It only failed in Many-shot by truncating the output to a single line.

---

## 2. Reasoning & Mathematical Accuracy


**Qwen 0.5B:**  
**Failed.** The model guessed erratically. In Zero-shot, it predicted **2400 kcal**, and in Few-shot it jumped to **2582 kcal**.

**TinyLlama 1.1B:**  
**Failed.** In Zero-shot, it produced heavily rounded guesses (**1200 kcal**). In all other prompts, it simply copied the exact values from *Example 1* in the prompt.

**StableLM 2 Zephyr 1.6B:**  
**Closest to the actual results**  StableLM's One-shot attempt (**BMR: 1445 kcal**) was surprisingly close to the actual results (**1489 kcal**). However, its accuracy degraded significantly as more examples were added. So we should stick with zero or one shot

---

## 3. Prompt Stability 




**Qwen 0.5B:**  
**Unpredictable.** Adding examples caused the model to shift from simple textual responses to generating Python code (e.g., `def get_calories():`), completely losing the intended task.

**TinyLlama 1.1B:**  
**Highly Unstable.** Introducing examples caused catastrophic repetition loops.

**StableLM 2 Zephyr 1.6B:**  
**Degrades at Many-shot.** It performed flawlessly up to Few-shot prompting. However, the Many-shot prompt overloaded the model, causing it to output only one line (`TDEE (sedentary): 2,046 kcal`) and stop prematurely.

---

## 4. Speed 

**How fast did the model return a response?**

**TinyLlama 1.1B:**  
**Slow (~10s).** Responses averaged around **10–12 seconds**, mostly due to time spent generating infinite loops.

**Qwen 0.5B:**  
**Slow (~14.5s).** While consistent, it was slow because it kept generating unwanted code and explanations.

**StableLM 2 Zephyr 1.6B:**  
**Blazing Fast (~1.5s).** It averaged under **2 seconds** for successful generations, making it highly viable for real-time applications.

### Model Selection

From the evaluation, we decided to use the `stabilityai/stablelm-2-zephyr-1_6b` model since it produces the most accurate results, with excellent formatting, the highest speed (~1.5 seconds per prompt), and strong stability.

However, we should limit prompts to **zero-shot or one-shot** settings, since using higher-shot prompts may lead to formatting errors and less accurate results.

In [15]:
def zero_shot(prompt):
    return f"""
{prompt}
Respond in this exact format:
BMR: X kcal
TDEE (sedentary): X kcal
TDEE (moderately active): X kcal
TDEE (very active): X kcal

Output:
"""

def one_shot(prompt):
    return f"""

You are a calorie estimation assistant. Use the Mifflin-St Jeor equation.
Always return all four activity levels in the exact format below.

Example:
Input: 25-year-old man, 80 kg, 175 cm
Output:
  BMR: 1,877 kcal
  TDEE (sedentary): 2,252 kcal
  TDEE (moderately active): 2,598 kcal
  TDEE (very active): 2,944 kcal

Now answer:
{prompt}
Output:

"""




In [16]:



model_id =   "stabilityai/stablelm-2-zephyr-1_6b"
 



based_prompt = "How many calories does a average 35-year-old woman who weighs 70kg and is 180 cm tall need per day?"
prompt_to_be_use = zero_shot(based_prompt)

out_put = {}


print(f"\n--- Testing Model: {model_id} ---")
try:
    


    
        
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, 
        device_map="auto", 
        dtype=torch.float16,
        trust_remote_code=False
    )



    
    
    
    
    
        
    inputs = tokenizer(prompt_to_be_use, return_tensors="pt").to(model.device)
    
    start_time = time.time()
    
    outputs = model.generate(**inputs, max_new_tokens=300) 
    end_time = time.time()
    
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    out_put = {
        "prompt": prompt_to_be_use,
        "response": response,
        "time_taken": round(end_time - start_time, 2)
        
    }


    print(f"Response: {response}\nTime: {round(end_time - start_time, 2)} \n")
    print("---------------------------------------------------------------------------------------------------")
        
except Exception as e:
    print(f"Failed to load or run {model_id}: {e}")
    
finally:

    if 'model' in locals():
        del model
    if 'tokenizer' in locals():
        del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


--- Testing Model: stabilityai/stablelm-2-zephyr-1_6b ---


Setting `pad_token_id` to `eos_token_id`:100257 for open-end generation.


Response: 
How many calories does a average 35-year-old woman who weighs 70kg and is 180 cm tall need per day?
Respond in this exact format:
BMR: X kcal
TDEE (sedentary): X kcal
TDEE (moderately active): X kcal
TDEE (very active): X kcal

Output:
BMR: 1360 kcal
TDEE (sedentary): 1800 kcal
TDEE (moderately active): 2160 kcal
TDEE (very active): 2520 kcal
Time: 2.27 

---------------------------------------------------------------------------------------------------
